# Example: Banknote Authentication using SVM Classification
In this example, we use a soft-margin support vector machine (SVM) with a radial basis function (RBF) kernel to classify the [UCI banknote authentication dataset](https://archive.ics.uci.edu/dataset/267/banknote+authentication). We rely on the [`LIBSVM.jl`](https://github.com/JuliaML/LIBSVM.jl) package for training and prediction. See the lecture notebook [▶ Support Vector Machines (SVM)](CHEME-152-M2-Lecture-SupportVector-Classification-Watch-Studio.ipynb) for the theoretical background.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * __Prepare the banknote dataset:__ Load the UCI banknote authentication dataset and split it into training and test sets using a fixed random seed for reproducibility.
> * __Train a kernelized SVM with `LIBSVM.jl`:__ Fit a soft-margin SVM with an RBF kernel on the augmented training examples and inspect the number of support vectors.
> * __Evaluate classification performance:__ Predict labels on the held-out test set, compute the confusion matrix, and interpret the resulting accuracy.

Let's get started!
___

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, loading any needed resources, such as sample datasets, and setting up any required constants.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages (including `LIBSVM.jl`, `CSV`, and `DataFrames`), and includes local source files in `src/`. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

___
## Task 1: Load and split the banknote dataset
In this task, we load the [banknote authentication dataset](https://archive.ics.uci.edu/dataset/267/banknote+authentication) into a `DataFrame`, convert it to a numeric matrix, and split it into training and test subsets. The training subset is used to fit the SVM in Task 2, and the test subset is used to evaluate generalization performance in Task 3.

> __About the dataset__
>
> The [banknote authentication dataset](https://archive.ics.uci.edu/dataset/267/banknote+authentication) contains `1372` instances. Each instance is described by four continuous wavelet-transform statistics of a banknote image — variance, skewness, curtosis, and entropy — together with a binary class label. The UCI CSV uses the spelling `curtosis` (without the `k`), so this is the column name you will see in the loaded `DataFrame`. The class labels in the loaded CSV here are remapped from the original $\{0,1\}$ encoding to $\{-1,1\}$, where $-1$ indicates a genuine banknote and $1$ indicates a forgery.

Load the CSV into the `df_banknote::DataFrame` variable.

In [2]:
df_banknote = CSV.read(joinpath(_PATH_TO_DATA, "data-banknote-authentication.csv"), DataFrame)

Row,variance,skewness,curtosis,entropy,class
,Float64,Float64,Float64,Float64,Int64
1,3.6216,8.6661,-2.8073,-0.44699,-1
2,4.5459,8.1674,-2.4586,-1.4621,-1
3,3.866,-2.6383,1.9242,0.10645,-1
4,3.4566,9.5228,-4.0112,-3.5944,-1
5,0.32924,-4.4552,4.5718,-0.9888,-1
6,4.3684,9.6718,-3.9606,-3.1625,-1
7,3.5912,3.0129,0.72888,0.56421,-1
8,2.0922,-6.81,8.4636,-0.60216,-1
9,3.2032,5.7588,-0.75345,-0.61251,-1


Next, convert the `DataFrame` to a numeric matrix `D_banknote::Matrix{Float64}` and choose how many of the `1372` instances to use for training. We use `1000` training examples here, leaving the remaining `372` as the test set.

In [ ]:
D_banknote = Matrix(df_banknote); # convert the DataFrame to a numeric matrix
number_of_training_examples_banknote = 1000; # number of training examples

We split the data into training and test subsets using a fixed random seed so the split is reproducible across runs. The `randperm(n)` call returns a permutation of `1:n`; the first `number_of_training_examples_banknote` entries are used for training and the remainder for testing.

In [ ]:
using Random

banknote_training, banknote_test = let

    Random.seed!(42); # fixed seed for reproducibility
    n = size(D_banknote, 1);
    perm = randperm(n);
    train_idx = perm[1:number_of_training_examples_banknote];
    test_idx  = perm[number_of_training_examples_banknote+1:end];

    D_banknote[train_idx, :], D_banknote[test_idx, :]
end;

___
## Task 2: Train the soft-margin SVM
In this task, we fit a soft-margin SVM with an RBF kernel on the training subset using the [`svmtrain(...)` method](https://github.com/JuliaML/LIBSVM.jl) from the `LIBSVM.jl` package.

> __About the `svmtrain` interface__
>
> The [`svmtrain(...)` method](https://github.com/JuliaML/LIBSVM.jl) takes an augmented feature matrix $\hat{\mathbf{X}}^{\top}\in\mathbb{R}^{p\times n}$ (features along the rows, examples along the columns) and a label vector $\mathbf{y}\in\{-1,1\}^{n}$. It returns a trained model containing the learned dual variables, the support vectors, and the kernel hyperparameters. The `kernel` keyword selects from a family of [supported kernels](https://en.wikipedia.org/wiki/Support_vector_machine#Nonlinear_kernels); here we use the radial basis function (RBF) kernel.

Build the augmented training matrix `X`, the training label vector `y`, and call `svmtrain(...)` with `verbose = true` so the optimizer prints iteration and support-vector statistics during training.

In [ ]:
model = let

    D = banknote_training;
    n = size(D, 1);
    X = [D[:, 1:end-1] ones(n)] |> transpose |> Matrix; # augmented features (p × n)
    y = D[:, end]; # labels in {-1, 1}

    svmtrain(X, y, kernel = LIBSVM.Kernel.RadialBasis, verbose = true)
end;

In [ ]:
println("SVM type: $(model.SVMtype)")
println("Kernel:   $(model.kernel)")
println("Number of support vectors: $(length(model.SVs.indices))")
println("Penalty parameter C: $(model.cost)")

___
## Task 3: Evaluate on the test set
In this task, we apply the trained model to the held-out test set and quantify how well it generalizes to data it has not seen. We use the [`svmpredict(...)` method](https://github.com/JuliaML/LIBSVM.jl) to obtain predicted labels, and we compare them to the true test labels using a confusion matrix.

The [`svmpredict(...)` method](https://github.com/JuliaML/LIBSVM.jl) returns the predicted labels (stored in `ŷ::Vector{Float64}`) and the corresponding decision values. We store the true test labels in `y::Vector{Float64}`.

In [ ]:
ŷ, y = let

    D = banknote_test;
    n = size(D, 1);
    X = [D[:, 1:end-1] ones(n)] |> transpose |> Matrix; # augmented features (p × n)
    y = D[:, end]; # true labels in {-1, 1}

    ŷ, _ = svmpredict(model, X);
    ŷ, y
end;

### Confusion Matrix
The confusion matrix is a $2\times 2$ table whose entries are the counts of true positives (TP), false positives (FP), true negatives (TN), and false negatives (FN) produced by the classifier on the test set. We compute it using the [`confusion(...)` method](src/Compute.jl) and store the result in `CM::Matrix{Int64}`.

In [ ]:
CM = confusion(y, ŷ)

In [ ]:
n_test = length(y);
n_correct = CM[1,1] + CM[2,2];
accuracy = n_correct / n_test;
println("Accuracy: $(accuracy) | Error: $(1 - accuracy)")

The classifier predicts every test instance correctly on this split. The banknote authentication dataset is known to be nearly separable in its wavelet-transform feature space, so a kernelized SVM is able to recover a near-perfect decision boundary. A different random seed will reshuffle the split but the accuracy on this dataset remains very high.
___
## Summary
In this example, we trained a soft-margin SVM with an RBF kernel on the UCI banknote authentication dataset and evaluated its accuracy on a held-out test set.

> __Key Takeaways:__
>
> * __Soft-margin SVM with `LIBSVM.jl`:__ The `svmtrain(...)` method fits a kernelized soft-margin SVM directly from the augmented training matrix and label vector, returning a model that records the support vectors and learned hyperparameters.
> * __Reproducible train/test splits matter:__ Using `Random.seed!` together with `randperm` gives a deterministic split, so the reported accuracy can be reproduced across runs.
> * __Confusion matrix summarizes classifier performance:__ The four entries of the confusion matrix capture the counts of correct and incorrect predictions in each direction, and their diagonal sum divided by the test-set size gives the overall accuracy.

These steps illustrate the typical workflow for applying a kernelized SVM to a binary classification problem.
___